# Regional airborne LiDAR pilot
Run this notebook on a CPU runtime. The worked example scans a 240 m square north of Bern, Switzerland. It uses swisstopo's 2023 swissSURFACE3D airborne returns in LV95 (EPSG:2056) and metre elevations. Model thresholds come from the Shizuoka research pipeline. Local detection sensitivity is unmeasured. A lead needs independent modern-land-use and field review.

[Source record](https://data.geo.admin.ch/api/stac/v1/collections/ch.swisstopo.swisssurface3d/items/swisssurface3d_2023_2598-1202) · [Project](https://github.com/wrong-nebula/palimpsest)

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile
root = Path('/content/palimpsest')
if not root.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', 'https://github.com/wrong-nebula/palimpsest.git', str(root)], check=True)
    subprocess.run(['git', '-C', str(root), 'sparse-checkout', 'set', 'palimpsest', 'notebooks', 'experiments'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root/'requirements-pilot.txt')], check=True)
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
asset = 'https://data.geo.admin.ch/ch.swisstopo.swisssurface3d/swisssurface3d_2023_2598-1202/swisssurface3d_2023_2598-1202_2056_5728.las.zip'
sha256 = '46e12d9ec3413f0ac3e08cf4d4775a1e7b94d09444c6ef0c95f334b54614715a'
record = 'https://data.geo.admin.ch/api/stac/v1/collections/ch.swisstopo.swisssurface3d/items/swisssurface3d_2023_2598-1202'
archive = Path('/content/swisssurface3d.zip')
if not archive.exists():
    tmp = archive.with_suffix('.part')
    urllib.request.urlretrieve(asset, tmp)
    tmp.replace(archive)
with archive.open('rb') as f:
    assert hashlib.file_digest(f, 'sha256').hexdigest() == sha256, 'Source archive checksum mismatch'
source = Path('/content/swisssurface3d.las')
with zipfile.ZipFile(archive) as z, z.open('2598_1202.las') as src, source.open('wb') as dst:
    shutil.copyfileobj(src, dst)
epsg = 2056
bounds = [2598280, 1202700, 2598520, 1202940]

### Scan one footprint
For a different survey, set `source` to a LAS/LAZ file, `record` to its source citation, `epsg` to its projected metre CRS and `bounds` to the west, south, east and north edges. Check vertical units in the survey metadata. Include a 50 m halo on every side, and use a core up to 300 m wide. Regional scans export ground support and unreviewed leads. The Japanese aerial-history classes only apply to the Shizuoka dossier.

In [ ]:
import tempfile
out = Path(tempfile.mkdtemp(prefix='regional-', dir='/content'))
subprocess.run([sys.executable, '-m', 'palimpsest.regional_pilot', str(source), '--source-ref', record,
                '--epsg', str(epsg), '--bounds', *map(str, bounds), '--out', str(out)], cwd=root, check=True)
print((out/'scan.json').read_text())

In [ ]:
from google.colab import files
bundle = shutil.make_archive(str(out), 'zip', out)
files.download(bundle)